# Experience memory validation

This notebook runs the repository's existing behavior tests for experience memory and outcome integrity. Saved command output and execution provenance occupy one artifact. The notebook calls the Rust tests directly and contains no duplicate test logic.

Execute it from the repository root after the code is frozen:

```bash
.venv/bin/jupyter execute evals/bench/experience-memory/validation.ipynb --inplace --timeout=900
```

Treat this file as unexecuted evidence until every code cell has an execution count, saved output, and no error. The provenance cell identifies the Git working tree and binary used for an executed result.

## Validation questions

The commands test four claims:

1. A failed attempt can be followed by a repaired file, a passing check, and conditional lesson recall in a later session.
2. Recall abstains when machine, version, or symptom conditions do not match.
3. Passive literal reuse is recorded as `observed_used`; it does not imply success and cannot alter ranking priors. Historical scanner-generated `accepted` rows remain readable but do not affect priors.
4. Check receipts store output hashes without output bodies. Source changes prevent lesson support, timeout and interruption stop descendants, and invalid targets fail before the checker runs.

In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import shlex
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import display as show


def find_repo_root(start: Path) -> Path:
    """Find the memd checkout above the notebook kernel directory."""
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").is_file() and (
            candidate / "crates" / "memd"
        ).is_dir():
            return candidate
    raise RuntimeError(f"cannot find the memd repository above {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
STARTED_AT_UTC = datetime.now(UTC)
TEST_ENV = os.environ.copy()
TEST_ENV["CARGO_BUILD_JOBS"] = "8"
TEST_ENV["RUSTC_WRAPPER"] = ""
TEST_ENV["TMPDIR"] = "/var/tmp"


def run_command(name: str, command: list[str], *, timeout_s: int = 900) -> str:
    """Run one repository gate and return its combined output."""
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=TEST_ENV,
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout_s,
    )
    output = completed.stdout
    print(f"[{name}] $ {shlex.join(command)}")
    print(output)
    if completed.returncode != 0:
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}")
    return output


def read_command(command: list[str]) -> str:
    """Read a short command result for the provenance record."""
    return subprocess.run(
        command,
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


def sha256_file(path: Path) -> str:
    """Return the SHA-256 digest of one file."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


{
    "repository_root": str(REPO_ROOT),
    "started_at_utc": STARTED_AT_UTC.isoformat(),
    "cargo_build_jobs": TEST_ENV["CARGO_BUILD_JOBS"],
}

{'repository_root': '/home/fschulz/dev/memd-worktrees/experience-memory',
 'started_at_utc': '2026-09-16T06:30:37.649272+00:00',
 'cargo_build_jobs': '8'}

## Run the behavior gates

The first command exercises the public CLI and domain model. The next three commands test the outcome-kind contract, current Codex log parsing with native action time, and the persisted legacy-row prior guard. All commands use Rust 1.98.0 and at most eight Cargo build jobs.

In [2]:
TEST_COMMANDS = {
    "experience_cli_and_domain": [
        "cargo",
        "+1.98.0",
        "test",
        "-p",
        "memd",
        "--test",
        "experience_cli",
        "--test",
        "experience_memory",
        "--",
        "--nocapture",
    ],
    "observed_use_contract": [
        "cargo",
        "+1.98.0",
        "test",
        "-p",
        "memd",
        "--lib",
        "store::outcome::tests::observed_use_and_legacy_scanner_events_never_credit_success",
        "--",
        "--exact",
        "--nocapture",
    ],
    "current_codex_log_contract": [
        "cargo",
        "+1.98.0",
        "test",
        "-p",
        "memd",
        "--lib",
        "cli::outcome_scan::tests::current_custom_tool_reuse_records_native_time_once_after_append",
        "--",
        "--exact",
        "--nocapture",
    ],
    "persisted_legacy_prior_guard": [
        "cargo",
        "+1.98.0",
        "test",
        "-p",
        "memd",
        "--test",
        "outcome_attribution",
        "legacy_codex_scanner_acceptance_does_not_enter_persistent_priors",
        "--",
        "--exact",
        "--nocapture",
    ],
}

outputs_by_name = {
    name: run_command(name, command) for name, command in TEST_COMMANDS.items()
}

[experience_cli_and_domain] $ cargo +1.98.0 test -p memd --test experience_cli --test experience_memory -- --nocapture
   Compiling memd v1.7.1 (/home/fschulz/dev/memd-worktrees/experience-memory/crates/memd)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 6.53s
     Running tests/experience_cli.rs (target/debug/deps/experience_cli-69a513e81bf640cb)

running 2 tests
test interrupted_and_invalid_checks_stop_before_leaking_side_effects ... ok
EXPERIENCE_SIMULATION_RESULT={"attempt_count":4,"check_count":4,"exported_artifact_count":12,"imported_artifact_count":12,"native_thread_count":2,"problem_count":3,"recall_abstention_count":3,"recall_match_count":1,"replayed_artifact_count":12,"scenario":"repair_recall_privacy_timeout_transfer","supported_lesson_count":1,"timeout_seconds":1}
test repair_recall_privacy_timeout_and_transfer_work_through_cli ... ok

test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 2.33s

     Running tests/exper

[observed_use_contract] $ cargo +1.98.0 test -p memd --lib store::outcome::tests::observed_use_and_legacy_scanner_events_never_credit_success -- --exact --nocapture
   Compiling memd v1.7.1 (/home/fschulz/dev/memd-worktrees/experience-memory/crates/memd)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 12.06s
     Running unittests src/lib.rs (target/debug/deps/memd-190ca4861a6145f8)

running 1 test
test store::outcome::tests::observed_use_and_legacy_scanner_events_never_credit_success ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 1045 filtered out; finished in 0.00s




[current_codex_log_contract] $ cargo +1.98.0 test -p memd --lib cli::outcome_scan::tests::current_custom_tool_reuse_records_native_time_once_after_append -- --exact --nocapture
    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.19s
     Running unittests src/lib.rs (target/debug/deps/memd-190ca4861a6145f8)

running 1 test
test cli::outcome_scan::tests::current_custom_tool_reuse_records_native_time_once_after_append ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 1045 filtered out; finished in 0.01s




[persisted_legacy_prior_guard] $ cargo +1.98.0 test -p memd --test outcome_attribution legacy_codex_scanner_acceptance_does_not_enter_persistent_priors -- --exact --nocapture
   Compiling memd v1.7.1 (/home/fschulz/dev/memd-worktrees/experience-memory/crates/memd)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 2.18s
     Running tests/outcome_attribution.rs (target/debug/deps/outcome_attribution-69c21c0b3ba84daf)

running 1 test
test legacy_codex_scanner_acceptance_does_not_enter_persistent_priors ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 12 filtered out; finished in 0.05s




## Inspect the end-to-end result

The CLI test emits one machine-readable line after its assertions pass. The counts describe this deterministic fixture. They do not estimate autonomous-agent accuracy or utility.

In [3]:
SIMULATION_PREFIX = "EXPERIENCE_SIMULATION_RESULT="
simulation_lines = [
    line.removeprefix(SIMULATION_PREFIX)
    for line in outputs_by_name["experience_cli_and_domain"].splitlines()
    if line.startswith(SIMULATION_PREFIX)
]
if len(simulation_lines) != 1:
    raise RuntimeError(
        f"expected one simulation result line, found {len(simulation_lines)}"
    )

simulation = json.loads(simulation_lines[0])
expected_counts = {
    "problem_count": 3,
    "attempt_count": 4,
    "check_count": 4,
    "supported_lesson_count": 1,
    "recall_match_count": 1,
    "recall_abstention_count": 3,
    "native_thread_count": 2,
    "exported_artifact_count": 12,
    "imported_artifact_count": 12,
    "replayed_artifact_count": 12,
}
for field, expected in expected_counts.items():
    observed = simulation.get(field)
    if observed != expected:
        raise RuntimeError(f"{field}: expected {expected}, observed {observed}")

show(simulation)

{'attempt_count': 4,
 'check_count': 4,
 'exported_artifact_count': 12,
 'imported_artifact_count': 12,
 'native_thread_count': 2,
 'problem_count': 3,
 'recall_abstention_count': 3,
 'recall_match_count': 1,
 'replayed_artifact_count': 12,
 'scenario': 'repair_recall_privacy_timeout_transfer',
 'supported_lesson_count': 1,
 'timeout_seconds': 1}

## Record execution provenance

The provenance record binds the saved outputs to the repository revision, dirty state, selected tool versions, host, and built `memd` binary. It lists changed paths without recording file contents or environment secrets.

In [4]:
binary_path = REPO_ROOT / "target" / "debug" / "memd"
if not binary_path.is_file():
    raise RuntimeError(f"test build did not produce {binary_path}")

git_status = read_command(["git", "status", "--porcelain=v1"])
provenance = {
    "execution_started_at_utc": STARTED_AT_UTC.isoformat(),
    "execution_finished_at_utc": datetime.now(UTC).isoformat(),
    "host": platform.node(),
    "platform": platform.platform(),
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "kernel": "python3",
    "ipykernel_version": importlib.metadata.version("ipykernel"),
    "nbclient_version": importlib.metadata.version("nbclient"),
    "nbformat_version": importlib.metadata.version("nbformat"),
    "rustc_version": read_command(["rustc", "+1.98.0", "--version"]),
    "cargo_version": read_command(["cargo", "+1.98.0", "--version"]),
    "git_revision": read_command(["git", "rev-parse", "HEAD"]),
    "git_dirty": bool(git_status),
    "git_status_porcelain": git_status.splitlines(),
    "binary_path": str(binary_path.relative_to(REPO_ROOT)),
    "binary_version": read_command([str(binary_path), "--version"]),
    "binary_sha256": sha256_file(binary_path),
    "test_environment": {
        "CARGO_BUILD_JOBS": TEST_ENV["CARGO_BUILD_JOBS"],
        "RUSTC_WRAPPER": TEST_ENV["RUSTC_WRAPPER"],
        "TMPDIR": TEST_ENV["TMPDIR"],
    },
    "commands": TEST_COMMANDS,
}
show(provenance)

{'execution_started_at_utc': '2026-09-16T06:30:37.649272+00:00',
 'execution_finished_at_utc': '2026-09-16T06:31:01.328943+00:00',
 'host': 'nelli-gpu',
 'platform': 'Linux-7.0.0-22-generic-x86_64-with-glibc2.43',
 'python_executable': '/home/fschulz/dev/memd-worktrees/experience-memory/.venv/bin/python',
 'python_version': '3.12.13',
 'kernel': 'python3',
 'ipykernel_version': '7.3.0',
 'nbclient_version': '0.11.0',
 'nbformat_version': '5.11.1',
 'rustc_version': 'rustc 1.98.0 (88d9e12ae 2026-08-18)',
 'cargo_version': 'cargo 1.98.0 (797e8a9bc 2026-08-05)',
 'git_revision': '232986161717055d6213ead3b3a6c3152ae43e80',
 'git_dirty': True,
 'git_status_porcelain': ['M  README.md',
  'M  crates/memd/Cargo.toml',
  'M  crates/memd/src/cli/args.rs',
  'M  crates/memd/src/cli/batch.rs',
  'A  crates/memd/src/cli/experience.rs',
  'A  crates/memd/src/cli/experience_check.rs',
  'M  crates/memd/src/cli/mod.rs',
  'M  crates/memd/src/cli/ops_bridge.rs',
  'M  crates/memd/src/cli/outcome_scan.r

## Interpretation and limits

A passing run supports the specified behavior on one machine with deterministic fixtures and temporary persistent stores. The end-to-end scenario uses two stores on that machine to test exact export, import, and replay. It also sends two native thread identities through one warm worker and inspects their stored attribution.

This notebook does not measure autonomous-agent utility, success rate, retrieval accuracy, or multi-machine synchronization. The passive scanner establishes later literal reuse only. It provides no evidence that the reused action succeeded. These tests do not cover production workloads, other operating systems, or uncontrolled concurrent clients.